[![img/cloudevel.png](img/cloudevel.png)](https://cloudevel.com)

# Combinación de tablas y lógica condicional.

**sql101 · SQL para Ingeniería y Analítica de Datos** · Cloudevel® · Instructor: José Luis Chiquete · Módulo 2: *Resumir y combinar*

> 📝 **Versión del alumno.** Las celdas marcadas con ```-- TU CÓDIGO AQUÍ``` y las de *Tu respuesta* son para ti.

En este *notebook* se combinarán tablas con ```JOIN``` y se aplicará lógica condicional con ```CASE```.

## 🧭 Cómo usar este *notebook*.

* Ejecuta primero la celda de **configuración** y después las demás **en orden** (```Shift + Enter```).
* Las celdas que comienzan con ```%%sql``` son consultas: todo lo que sigue a ```%%sql``` es *SQL* y puede copiarse a otro motor.
* **Ejercicio:** lo que todos deben dominar. **Reto (opcional):** para quien quiere profundizar; se puede resolver como práctica.
* En las celdas ```-- TU CÓDIGO AQUÍ```, reemplaza la línea ```SELECT NULL AS tu_respuesta;``` por tu consulta. Las de *✍️ Tu respuesta* son para escribir.

## Configuración.

Abre la base de datos del taller en modo de sólo lectura y habilita las celdas ```%%sql```.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/thelook.duckdb', read_only=True)
%sql conexion

## ¿Por qué los datos están en varias tablas?

Los sistemas transaccionales (*OLTP*) guardan cada entidad en su propia tabla para no repetir información: los datos de un cliente se guardan una sola vez en ```users``` y cada orden sólo hace referencia a él mediante ```user_id```. A este diseño se le conoce como **normalización**.

* Una **clave primaria** identifica de forma única cada fila de una tabla (```users.id```).
* Una **clave foránea** es una columna que apunta a la clave primaria de otra tabla (```orders.user_id```).

```
users (id) ──< orders (order_id, user_id) ──< order_items (order_id, product_id) >── products (id) >── distribution_centers (id)
```

Para responder preguntas que involucran varias entidades (por ejemplo, *ventas por categoría de producto*), es necesario **combinar** las tablas con ```JOIN```.

## ```INNER JOIN```.

```INNER JOIN``` regresa sólo las filas que tienen coincidencia en ambas tablas, según la condición de ```ON```. Se acostumbra usar alias cortos para las tablas.

* Cada artículo vendido con el nombre y la categoría del producto.

In [ ]:
%%sql
SELECT oi.order_id,
  oi.sale_price,
  p.name,
  p.category
FROM order_items AS oi
INNER JOIN products AS p
  ON oi.product_id = p.id
ORDER BY oi.order_id
FETCH FIRST 20 ROWS ONLY;

* Combinado con agregaciones: ventas por categoría.

In [ ]:
%%sql
SELECT p.category,
  COUNT(*) AS articulos,
  ROUND(SUM(oi.sale_price), 2) AS ventas
FROM order_items AS oi
JOIN products AS p
  ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY p.category
ORDER BY ventas DESC;

**Nota:** ```JOIN``` sin calificativo equivale a ```INNER JOIN```.

### Más de dos tablas.

* Ventas por país del cliente y centro de distribución del producto.

In [ ]:
%%sql
SELECT u.country,
  dc.name AS centro_distribucion,
  ROUND(SUM(oi.sale_price), 2) AS ventas
FROM order_items AS oi
JOIN users AS u
  ON oi.user_id = u.id
JOIN products AS p
  ON oi.product_id = p.id
JOIN distribution_centers AS dc
  ON p.distribution_center_id = dc.id
GROUP BY u.country, dc.name
ORDER BY ventas DESC
FETCH FIRST 20 ROWS ONLY;

## ```LEFT JOIN```.

```LEFT JOIN``` conserva **todas** las filas de la tabla de la izquierda; cuando no hay coincidencia, las columnas de la derecha quedan en ```NULL```. Es la herramienta para encontrar "lo que falta".

* Clientes registrados que nunca han hecho una orden.

In [ ]:
%%sql
SELECT u.id, u.first_name, u.last_name, u.created_at
FROM users AS u
LEFT JOIN orders AS o
  ON u.id = o.user_id
WHERE o.order_id IS NULL
ORDER BY u.id
FETCH FIRST 20 ROWS ONLY;

* Número de órdenes por cliente, incluyendo a quienes tienen cero. ```COUNT(o.order_id)``` cuenta sólo los valores no nulos.

In [ ]:
%%sql
SELECT u.id,
  u.first_name,
  COUNT(o.order_id) AS ordenes
FROM users AS u
LEFT JOIN orders AS o
  ON u.id = o.user_id
GROUP BY u.id, u.first_name
ORDER BY ordenes DESC, u.id
FETCH FIRST 20 ROWS ONLY;

### Otros tipos de ```JOIN```.

* ```RIGHT JOIN```: el inverso de ```LEFT JOIN```; casi siempre se reescribe como ```LEFT JOIN``` invirtiendo las tablas.
* ```FULL OUTER JOIN```: conserva todas las filas de ambas tablas. Útil para conciliar dos fuentes.
* ```CROSS JOIN```: todas las combinaciones posibles entre filas.

https://cloud.google.com/bigquery/docs/reference/standard-sql/query-syntax#join_types

### Cuidado: duplicación de filas.

Cuando una fila de la izquierda coincide con varias de la derecha, se repite. Si se combina ```orders``` con ```order_items``` y después se suma una columna de ```orders``` (como ```num_of_item```), el valor se sumará una vez por cada artículo. **Antes de combinar, identifica la granularidad de cada tabla: qué representa una fila.**

## Lógica condicional con ```CASE```.

```CASE``` evalúa condiciones en orden y regresa el valor de la primera que se cumple; ```ELSE``` define el valor por omisión.

* Clasificación de productos por rango de precio.

In [ ]:
%%sql
SELECT name,
  retail_price,
  CASE
    WHEN retail_price < 25 THEN 'Económico'
    WHEN retail_price < 75 THEN 'Medio'
    ELSE 'Premium'
  END AS segmento
FROM products
ORDER BY retail_price DESC
FETCH FIRST 20 ROWS ONLY;

* ```CASE``` dentro de ```GROUP BY```: clientes por grupo de edad.

In [ ]:
%%sql
SELECT
  CASE
    WHEN age < 25 THEN '18-24'
    WHEN age < 35 THEN '25-34'
    WHEN age < 50 THEN '35-49'
    ELSE '50+'
  END AS grupo_edad,
  COUNT(*) AS clientes
FROM users
GROUP BY
  CASE
    WHEN age < 25 THEN '18-24'
    WHEN age < 35 THEN '25-34'
    WHEN age < 50 THEN '35-49'
    ELSE '50+'
  END
ORDER BY grupo_edad;

> **Nota de dialecto:** el estándar no permite usar en ```GROUP BY``` un alias definido en ```SELECT```, por eso se repite la expresión. *BigQuery*, *PostgreSQL* y otros sí lo permiten (```GROUP BY grupo_edad```). En la siguiente lección se verá cómo evitar la repetición con ```WITH```.

* ```CASE``` dentro de una agregación (agregación condicional): convierte valores de filas en columnas (*pivote*). Ventas completadas, devueltas y tasa de devolución por categoría:

In [ ]:
%%sql
SELECT p.category,
  ROUND(SUM(CASE WHEN oi.status = 'Complete' THEN oi.sale_price ELSE 0 END), 2) AS ventas_completadas,
  ROUND(SUM(CASE WHEN oi.status = 'Returned' THEN oi.sale_price ELSE 0 END), 2) AS ventas_devueltas,
  ROUND(100.0 * COUNT(CASE WHEN oi.status = 'Returned' THEN 1 END)
        / COUNT(CASE WHEN oi.status IN ('Complete', 'Returned') THEN 1 END), 1) AS tasa_devolucion_pct
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.id
GROUP BY p.category
ORDER BY tasa_devolucion_pct DESC;

## ✏️ Ejercicios.

Los **ejercicios** cubren lo que todos deben dominar; los **retos** son opcionales. Las soluciones están en ```soluciones/```.

### Ejercicio 1 · Las marcas que más venden.

Obtén las 10 marcas con más ventas (```sale_price```) considerando sólo artículos con estado ```Complete```.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 2 · Productos sin ventas.

Lista los productos que nunca se han vendido (nombre y categoría).

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 3 · Ventas por origen.

Para cada ```traffic_source``` del cliente, calcula el número de órdenes y el total vendido.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Ejercicio 4 · Clasificar con CASE.

Clasifica las órdenes por número de artículos (```num_of_item```): ```'1 artículo'```, ```'2-3 artículos'``` y ```'4 o más'```, y cuenta cuántas hay en cada grupo.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ: reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

### Reto 1 · Pivote por centro de distribución.

Por centro de distribución, calcula el número de artículos vendidos con estado ```Complete``` y el número con estado ```Returned``` en una sola consulta.

In [ ]:
%%sql
-- TU CÓDIGO AQUÍ (opcional): reemplaza la siguiente línea por tu consulta.
SELECT NULL AS tu_respuesta;

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>